# Multiple simulation demo

This notebook runs many randomized 3-year simulations using:

- Equal-weight allocation across all available assets
- Annual rebalancing
- Dividend reinvestment

It then reports:

- Overall results
- Histogram of total returns
- Histogram of Sortino ratios
- Asset and portfolio value charts for best and worst runs

In [ ]:
from datetime import date
from importlib import reload

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns

from investing import history
from investing import history as history_mod
from investing import reporting
from investing import simulation
from investing.portfolio import AssetAllocation, HoldingTarget

reload(history)
reload(history_mod)
reload(reporting)
reload(simulation)

sns.set_theme(style="whitegrid")

In [ ]:
market_history = history_mod.load_market_history(
    "../data/5-way-prices.xlsx",
    "../data/5-way-dividends.xlsx",
)

sorted_tickers = sorted(market_history.securities.keys())
allocation = AssetAllocation([HoldingTarget(ticker, 1) for ticker in sorted_tickers])
strategy = simulation.AnnualRebalance(allocation, max_deviation=0.05)

sorted_tickers

In [ ]:
YEARS = 3
START_FUNDS = 100_000
NUM_SIMULATIONS = 50
SEED = 42
PLAN_TARGET_RETURN = 0.04

multi = simulation.simulate_many(
    strategy=strategy,
    history=market_history,
    years=YEARS,
    start_funds=START_FUNDS,
    num_simulations=NUM_SIMULATIONS,
    seed=SEED,
    show_progress=True,
)

len(multi.simulations)

In [ ]:
run_rows = []
for idx, (run, run_metrics) in enumerate(zip(multi.simulations, multi.run_metrics)):
    start_portfolio = run.portfolios[0]
    end_portfolio = run.portfolios[-1]

    start_value = float(
        start_portfolio.total_value(start_portfolio.as_of_date, market_history)
    )
    end_value = float(end_portfolio.total_value(end_portfolio.as_of_date, market_history))
    total_return = end_value / start_value - 1.0

    run_rows.append(
        {
            "run": idx,
            "start_date": start_portfolio.as_of_date,
            "end_date": end_portfolio.as_of_date,
            "start_value": start_value,
            "end_value": end_value,
            "total_return": total_return,
            "sortino_ratio": run_metrics.sortino_ratio,
            "cagr": run_metrics.cagr,
            "max_drawdown": run_metrics.max_drawdown,
            "std_dev_returns": run_metrics.std_dev_returns,
        }
    )

run_results = pl.DataFrame(run_rows)
run_results.head()

In [ ]:
overall_results = {
    "num_simulations": len(multi.simulations),
    "years": YEARS,
    "start_funds": START_FUNDS,
    "plan_target_return": PLAN_TARGET_RETURN,
    "avg_total_return": float(run_results["total_return"].mean()),
    "median_total_return": float(run_results["total_return"].median()),
    "min_total_return": float(run_results["total_return"].min()),
    "max_total_return": float(run_results["total_return"].max()),
    "avg_sortino_ratio": float(run_results["sortino_ratio"].drop_nulls().mean()),
    "median_sortino_ratio": float(run_results["sortino_ratio"].drop_nulls().median()),
    "aggregate_cagr": multi.metrics.cagr,
    "aggregate_max_drawdown": multi.metrics.max_drawdown,
    "aggregate_std_dev_returns": multi.metrics.std_dev_returns,
    "aggregate_sortino_ratio": multi.metrics.sortino_ratio,
    "aggregate_success_probability": multi.metrics.success_probability,
    "terminal_wealth_p10": multi.metrics.terminal_wealth_p10,
    "terminal_wealth_p50": multi.metrics.terminal_wealth_p50,
    "terminal_wealth_p90": multi.metrics.terminal_wealth_p90,
}

pl.DataFrame([overall_results]).transpose(include_header=True, header_name="metric", column_names=["value"])

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(run_results, x="total_return", bins=30, kde=True)
plt.title("Distribution of 3-Year Total Returns")
plt.xlabel("Total return")
plt.ylabel("Simulation count")
plt.show()

In [ ]:
sortino_values = run_results.select("sortino_ratio").drop_nulls().to_pandas()

plt.figure(figsize=(10, 5))
sns.histplot(sortino_values, x="sortino_ratio", bins=30, kde=True)
plt.title("Distribution of Sortino Ratios")
plt.xlabel("Sortino ratio")
plt.ylabel("Simulation count")
plt.show()

In [ ]:
best_run_id = int(run_results.sort("end_value", descending=True)["run"][0])
worst_run_id = int(run_results.sort("end_value", descending=False)["run"][0])

best_values = reporting.value_history(
    multi.simulations[best_run_id].portfolios,
    market_history,
    "monthly",
)
worst_values = reporting.value_history(
    multi.simulations[worst_run_id].portfolios,
    market_history,
    "monthly",
)

plt.figure(figsize=(12, 5))
sns.lineplot(
    best_values.filter(pl.col("ticker") != "_TOTAL"),
    x="date",
    y="valuation",
    hue="ticker",
)
plt.title(f"Best run asset values (run {best_run_id})")
plt.xlabel("Date")
plt.ylabel("Value")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
sns.lineplot(
    best_values.filter(pl.col("ticker") == "_TOTAL"),
    x="date",
    y="valuation",
    color="black",
)
plt.title(f"Best run portfolio value (run {best_run_id})")
plt.xlabel("Date")
plt.ylabel("Total value")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
sns.lineplot(
    worst_values.filter(pl.col("ticker") != "_TOTAL"),
    x="date",
    y="valuation",
    hue="ticker",
)
plt.title(f"Worst run asset values (run {worst_run_id})")
plt.xlabel("Date")
plt.ylabel("Value")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
sns.lineplot(
    worst_values.filter(pl.col("ticker") == "_TOTAL"),
    x="date",
    y="valuation",
    color="black",
)
plt.title(f"Worst run portfolio value (run {worst_run_id})")
plt.xlabel("Date")
plt.ylabel("Total value")
plt.tight_layout()
plt.show()